# 07 — Results Visualization

All plots and statistical analyses for the cross-lingual ECoG encoding experiment.
Loads pre-computed encoding results and electrode selection from notebook 06.

## Figures produced

| Figure | Description | Key comparison |
|--------|-------------|----------------|
| 1 | Grand-average time-lag curves per mode | All modes, EN/HE/AR/noise |
| 2 | Model architecture comparison | XLM-R vs XGLM vs FastText on EN |
| 3 | Residual analysis across modes | EN alone vs EN+HE/AR residual |
| 4 | Cross-mode residual comparison | Does residual advantage vary by model? |
| 5 | Electrode encoding profile | Ranked electrode plot |
| 6 | Pre vs post onset | Predictive processing signal |
| 7 | Per-subject violin + jitter | Individual variability, n=9 |

## 1. Imports

In [ ]:
import os
import re
import glob
import json
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MultipleLocator
from scipy import stats
from scipy.ndimage import gaussian_filter1d

try:
    from statsmodels.stats.multitest import multipletests
    HAS_STATSMODELS = True
except ImportError:
    warnings.warn('statsmodels not found — FDR correction skipped.')
    HAS_STATSMODELS = False

warnings.filterwarnings('ignore')
print(f'NumPy {np.__version__}  Matplotlib {matplotlib.__version__}')

## 2. Configuration

In [ ]:
RESULTS_DIR = '../results/'
freq        = 64
tmin, tmax  = -2.0, 2.0

# ── Result folders (must match notebook 06 output structure) ─────────────────
FOLDERS = {
    'fasttext'      : os.path.join(RESULTS_DIR, f'encoding_fasttext_{freq}Hz_({tmin},{tmax})'),
    'sliding_window': os.path.join(RESULTS_DIR, f'encoding_sliding_window_{freq}Hz_({tmin},{tmax})'),
    'contextual'    : os.path.join(RESULTS_DIR, f'encoding_contextual_{freq}Hz_({tmin},{tmax})'),
    'xglm'          : os.path.join(RESULTS_DIR, f'encoding_xglm_{freq}Hz_({tmin},{tmax})'),
}

# Condition names per mode (must match notebook 06 naming)
CONDITIONS = {
    'fasttext'      : {'en':'en_ft',   'he':'he_ft',   'ar':'ar_ft',
                       'en_he_res':'en+he_ft_res', 'en_ar_res':'en+ar_ft_res', 'noise':'noise_ft'},
    'sliding_window': {'en':'en_sw',   'he':'he_sw',   'ar':'ar_sw',
                       'en_he_res':'en+he_sw_res', 'en_ar_res':'en+ar_sw_res', 'noise':'noise_sw'},
    'contextual'    : {'en':'en_ctx',  'he':'he_ctx',  'ar':'ar_ctx',
                       'en_he_res':'en+he_ctx_res','en_ar_res':'en+ar_ctx_res','noise':'noise_ctx'},
    'xglm'          : {'en':'en_xglm', 'he':'he_xglm', 'ar':'ar_xglm',
                       'en_he_res':'en+he_xglm_res','en_ar_res':'en+ar_xglm_res','noise':'noise_xglm'},
}

# Time axis
LAGS             = np.linspace(tmin, tmax, 256)
POST_ONSET_RANGE = (0.0, 2.0)
PRE_RANGE        = (-0.8, 0.0)
POST_RANGE       = (0.0,  0.8)

# Electrode selection threshold
THRESHOLD = 0.10

# ── Publication-quality plot style ───────────────────────────────────────────
# Clean, Nature-journal aesthetic: Helvetica-adjacent sans-serif, minimal chrome,
# generous whitespace, strong signal lines.
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : 'white',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.spines.left' : True,
    'axes.spines.bottom': True,
    'axes.linewidth'   : 0.8,
    'axes.labelsize'   : 10,
    'axes.titlesize'   : 11,
    'axes.titleweight' : 'semibold',
    'axes.titlepad'    : 8,
    'axes.labelpad'    : 5,
    'xtick.labelsize'  : 8.5,
    'ytick.labelsize'  : 8.5,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size' : 3.5,
    'ytick.major.size' : 3.5,
    'xtick.direction'  : 'out',
    'ytick.direction'  : 'out',
    'legend.fontsize'  : 8,
    'legend.frameon'   : False,
    'legend.handlelength': 1.6,
    'font.family'      : 'sans-serif',
    'font.sans-serif'  : ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size'        : 9,
    'lines.linewidth'  : 2.0,
    'lines.solid_capstyle': 'round',
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'savefig.facecolor': 'white',
    'pdf.fonttype'     : 42,  # embeds fonts properly for journals
})

# ── Colour system ─────────────────────────────────────────────────────────────
# Language colours: distinct, colorblind-safe (Okabe-Ito palette base)
LANG_COL = {
    'en'   : '#0077BB',   # blue
    'he'   : '#CC3311',   # red
    'ar'   : '#009988',   # teal
    'noise': '#BBBBBB',   # grey
}

# Mode colours: for model architecture comparison
MODE_COL = {
    'xglm'          : '#EE7733',   # amber — causal/autoregressive
    'contextual'    : '#0077BB',   # blue  — bidirectional sentence
    'sliding_window': '#AA3377',   # purple — bidirectional window
    'fasttext'      : '#BBBBBB',   # grey  — static
}

MODE_LABEL = {
    'xglm'          : 'XGLM-1.7B (causal)',
    'contextual'    : 'XLM-R (sentence ctx)',
    'sliding_window': 'XLM-R (sliding window)',
    'fasttext'      : 'FastText (static)',
}

SEM_ALPHA  = 0.15
DPI        = 300
FIG_DIR    = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print('Configuration ready.')
print(f'Figure output : {os.path.abspath(FIG_DIR)}')

## 3. Load Electrode Selection and Helpers

In [ ]:
# ── Load electrode selection saved by notebook 06 ────────────────────────────
sel_tag  = f'r{str(THRESHOLD).replace(".","p")}'
sel_path = os.path.join(RESULTS_DIR, f'selected_electrodes_{sel_tag}.json')

if os.path.exists(sel_path):
    with open(sel_path) as f:
        _raw = json.load(f)
    selected_electrodes = {k: np.array(v, dtype=int) for k, v in _raw.items()}
    subjects = sorted(selected_electrodes.keys())
    print(f'Loaded electrode selection: {len(subjects)} subjects')
    for s, idx in selected_electrodes.items():
        print(f'  Subject {s}: {len(idx)} selected electrodes')
else:
    # Fallback: recompute from sliding_window EN condition
    print(f'Selection file not found ({sel_path}) — recomputing...')
    post_mask_sel = (LAGS >= POST_ONSET_RANGE[0]) & (LAGS <= POST_ONSET_RANGE[1])
    selected_electrodes = {}
    sw_folder = FOLDERS['sliding_window']
    for fp in sorted(glob.glob(os.path.join(sw_folder, 'corrs subj=* - en_sw.npy'))):
        m = re.match(r'corrs subj=([\w]+) -', os.path.basename(fp))
        if not m: continue
        subj = m.group(1)
        data = np.load(fp).mean(0)
        peak = data[:, post_mask_sel].max(1)
        selected_electrodes[subj] = np.where(peak >= THRESHOLD)[0]
    subjects = sorted(selected_electrodes.keys())
    print(f'Recomputed: {len(subjects)} subjects')


# ── Core helper functions ─────────────────────────────────────────────────────

def _find_file(folder, cond_name, subj):
    path = os.path.join(folder, f'corrs subj={subj} - {cond_name}.npy')
    if os.path.exists(path): return path
    matches = sorted(glob.glob(os.path.join(folder, f'corrs subj={subj} - {cond_name}*.npy')))
    return matches[0] if matches else None


def load_curve(folder, cond_name, subj, sigma=2):
    fpath = _find_file(folder, cond_name, subj)
    if fpath is None: return None
    data  = np.load(fpath).mean(0)
    sel   = selected_electrodes.get(subj)
    if sel is None or len(sel) == 0: return None
    curve = data[sel, :].mean(0)
    return gaussian_filter1d(curve, sigma=sigma) if sigma > 0 else curve


def grand_avg(folder, cond_name, sigma=4):
    curves = [load_curve(folder, cond_name, s, sigma) for s in subjects]
    curves = [c for c in curves if c is not None]
    if not curves: return None, None, 0
    arr = np.stack(curves)
    return arr.mean(0), arr.std(0) / np.sqrt(len(arr)), len(arr)


def subject_peaks(folder, cond_name, lag_range=POST_ONSET_RANGE):
    mask = (LAGS >= lag_range[0]) & (LAGS <= lag_range[1])
    out  = []
    for subj in subjects:
        fpath = _find_file(folder, cond_name, subj)
        sel   = selected_electrodes.get(subj)
        if fpath is None or sel is None or len(sel) == 0: continue
        data = np.load(fpath).mean(0)
        out.append(float(data[sel, :][:, mask].max(1).mean()))
    return out


def cohens_d(a, b):
    d = np.asarray(a) - np.asarray(b)
    return d.mean() / (d.std(ddof=1) + 1e-12)


def sig_star(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def fdr(pvals):
    pv = [p for p in pvals if not np.isnan(p)]
    if HAS_STATSMODELS and pv:
        _, corr, _, _ = multipletests(pv, method='fdr_bh')
        return list(corr)
    return pv


def decorate(ax, ylabel='Pearson r', xlabel='Time lag (s)', vline=True):
    if vline:
        ax.axvline(0, color='#444444', lw=0.9, ls='--', alpha=0.6, zorder=1)
    ax.axhline(0, color='#CCCCCC', lw=0.6, zorder=0)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xlim(LAGS[0], LAGS[-1])
    ax.xaxis.set_minor_locator(MultipleLocator(0.5))


def save_fig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=DPI)
    fig.savefig(path.replace('.png', '.pdf'))
    print(f'  Saved → {path}')
    plt.show()


# Check which modes have data
AVAILABLE_MODES = [m for m, f in FOLDERS.items() if os.path.isdir(f)]
print(f'\nAvailable modes: {AVAILABLE_MODES}')

## Figure 1 — Grand-Average Time-Lag Curves by Mode

In [ ]:
"""One panel per available mode. Each panel shows EN, HE, AR, noise.
Layout adapts to however many modes are available."""

n_modes  = len(AVAILABLE_MODES)
ncols    = min(n_modes, 2)
nrows    = (n_modes + 1) // 2
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(5.5 * ncols, 3.8 * nrows),
                          sharey=False)
axes = np.array(axes).flatten()

for ax_i, mode in enumerate(AVAILABLE_MODES):
    ax     = axes[ax_i]
    folder = FOLDERS[mode]
    conds  = CONDITIONS[mode]
    color  = MODE_COL[mode]

    lang_specs = [
        ('en',    'English', LANG_COL['en'],    '-',  2.2),
        ('he',    'Hebrew',  LANG_COL['he'],    '-',  2.0),
        ('ar',    'Arabic',  LANG_COL['ar'],    '-',  2.0),
        ('noise', 'Noise',   LANG_COL['noise'], '--', 1.2),
    ]
    for key, label, col, ls, lw in lang_specs:
        mu, sem, n = grand_avg(folder, conds[key])
        if mu is None: continue
        ax.plot(LAGS, mu, color=col, ls=ls, lw=lw, label=f'{label} (n={n})', zorder=4)
        ax.fill_between(LAGS, mu - sem, mu + sem, color=col, alpha=SEM_ALPHA, zorder=3)

    # Mode label as coloured tag in top-left
    ax.text(0.02, 0.97, MODE_LABEL[mode], transform=ax.transAxes,
            fontsize=8.5, fontweight='semibold', color=color,
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color, lw=0.8, alpha=0.9))

    decorate(ax)
    ax.legend(loc='lower right', fontsize=7.5)

# Hide any unused panels
for ax in axes[n_modes:]:
    ax.set_visible(False)

fig.suptitle('Grand-Average Encoding Performance by Embedding Mode',
             fontsize=12, fontweight='semibold', y=1.01)
plt.tight_layout(h_pad=3, w_pad=3)
save_fig(fig, 'fig1_grand_avg_by_mode.png')

## Figure 2 — Model Architecture Comparison (English)

In [ ]:
"""Directly addresses the causal vs bidirectional question.
One curve per mode, all showing the English condition only.
Inspired by Moraleda et al. (2025) Fig 2 model comparison."""

fig, ax = plt.subplots(figsize=(6.5, 4.0))

order = ['xglm', 'contextual', 'sliding_window', 'fasttext']
ls_map = {'xglm': '-', 'contextual': '-', 'sliding_window': '-', 'fasttext': '--'}

peak_vals = {}
for mode in order:
    if mode not in AVAILABLE_MODES: continue
    folder = FOLDERS[mode]
    cond   = CONDITIONS[mode]['en']
    mu, sem, n = grand_avg(folder, cond)
    if mu is None: continue
    col = MODE_COL[mode]
    lbl = MODE_LABEL[mode]
    ax.plot(LAGS, mu, color=col, ls=ls_map[mode], lw=2.2,
            label=f'{lbl} (n={n})', zorder=4)
    ax.fill_between(LAGS, mu - sem, mu + sem, color=col, alpha=SEM_ALPHA, zorder=3)
    peak_vals[mode] = subject_peaks(folder, cond)

# Annotate N400 peak region
ax.axvspan(0.35, 0.45, color='#EEEEEE', alpha=0.7, zorder=0)
ax.text(0.40, ax.get_ylim()[1] * 0.92, 'N400', ha='center', va='top',
        fontsize=7.5, color='#888888', style='italic')

decorate(ax, ylabel='Encoding performance (Pearson r)')
ax.set_title('Model Architecture Comparison — English Condition', fontsize=11)
ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
save_fig(fig, 'fig2_model_architecture_comparison.png')

## Figure 3 — Residual Analysis per Mode

In [ ]:
"""EN alone vs EN+HE_residual vs EN+AR_residual per mode.
Core figure: does the foreign-language residual add unique neural signal?"""

n_modes  = len(AVAILABLE_MODES)
fig, axes = plt.subplots(1, n_modes, figsize=(5.0 * n_modes, 4.0), sharey=False)
if n_modes == 1: axes = [axes]

for ax, mode in zip(axes, AVAILABLE_MODES):
    folder = FOLDERS[mode]
    conds  = CONDITIONS[mode]

    specs = [
        ('en',        'EN alone',         LANG_COL['en'],    '-',  2.4),
        ('en_he_res', 'EN + HE residual', LANG_COL['he'],    '-',  1.8),
        ('en_ar_res', 'EN + AR residual', LANG_COL['ar'],    '-',  1.8),
        ('noise',     'Noise',            LANG_COL['noise'], '--', 1.2),
    ]
    for key, label, col, ls, lw in specs:
        mu, sem, n = grand_avg(folder, conds[key])
        if mu is None: continue
        ax.plot(LAGS, mu, color=col, ls=ls, lw=lw, label=f'{label} (n={n})', zorder=4)
        ax.fill_between(LAGS, mu - sem, mu + sem, color=col, alpha=SEM_ALPHA, zorder=3)

    ax.text(0.02, 0.97, MODE_LABEL[mode], transform=ax.transAxes,
            fontsize=8, fontweight='semibold', color=MODE_COL[mode],
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=MODE_COL[mode], lw=0.8, alpha=0.9))
    decorate(ax)
    ax.legend(loc='lower right', fontsize=7.5)

for ax in axes[1:]:
    ax.set_ylabel('')

fig.suptitle('Residual Analysis — Unique Neural Variance of Foreign Language',
             fontsize=12, fontweight='semibold', y=1.01)
plt.tight_layout(w_pad=3)
save_fig(fig, 'fig3_residual_analysis.png')

## Figure 4 — Cross-Mode Residual Comparison

In [ ]:
"""Key scientific question: does the residual advantage (EN+residual > EN alone)
differ across model architectures? Grouped bar chart with per-subject dots.
Each bar = mean(EN+residual) - mean(EN alone), across subjects."""

residual_delta = {}  # (mode, lang) -> list of per-subject deltas

for mode in AVAILABLE_MODES:
    folder = FOLDERS[mode]
    conds  = CONDITIONS[mode]
    en_peaks = subject_peaks(folder, conds['en'])
    for lang_key, lang_label in [('en_he_res', 'HE'), ('en_ar_res', 'AR')]:
        res_peaks = subject_peaks(folder, conds[lang_key])
        n = min(len(en_peaks), len(res_peaks))
        if n < 3: continue
        deltas = [r - e for r, e in zip(res_peaks[:n], en_peaks[:n])]
        residual_delta[(mode, lang_label)] = deltas

# Plot
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharey=True)
fig.suptitle('Residual Advantage by Model Architecture',
             fontsize=12, fontweight='semibold', y=1.02)

rng = np.random.default_rng(42)
for ax, (lang_label, lang_col) in zip(axes, [('HE', LANG_COL['he']), ('AR', LANG_COL['ar'])]):
    keys   = [m for m in AVAILABLE_MODES if (m, lang_label) in residual_delta]
    x      = np.arange(len(keys))
    deltas = [residual_delta[(m, lang_label)] for m in keys]
    means  = [np.mean(d) for d in deltas]
    sems   = [np.std(d, ddof=1) / np.sqrt(len(d)) for d in deltas]
    colors = [MODE_COL[m] for m in keys]

    bars = ax.bar(x, means, color=colors, width=0.55,
                  edgecolor='white', linewidth=0.8, zorder=3, alpha=0.88)
    ax.errorbar(x, means, yerr=sems, fmt='none',
                color='#333333', capsize=4, capthick=0.9, lw=1.0, zorder=4)

    # Individual subject dots
    for xi, d, col in zip(x, deltas, colors):
        jit = rng.uniform(-0.15, 0.15, len(d))
        ax.scatter(xi + jit, d, color=col, s=28, zorder=5,
                   edgecolors='white', lw=0.5, alpha=0.9)

    ax.axhline(0, color='#444444', lw=0.8, ls='-', alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([MODE_LABEL[m].split('(')[0].strip() for m in keys],
                        rotation=20, ha='right', fontsize=8.5)
    ax.set_title(f'EN + {lang_label} residual  −  EN alone',
                 fontsize=10, pad=6, color=lang_col, fontweight='semibold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel('Δ Peak Pearson r', fontsize=10)
plt.tight_layout(w_pad=3)
save_fig(fig, 'fig4_cross_mode_residual_comparison.png')

## Figure 5 — Electrode Encoding Profile (Ranked)

In [ ]:
"""All selected electrodes pooled across subjects, sorted by EN peak r
in the reference mode. Shows how all model types compare electrode-by-electrode."""

post_mask = (LAGS >= POST_ONSET_RANGE[0]) & (LAGS <= POST_ONSET_RANGE[1])

def elec_peaks_all(folder, cond_name):
    out = []
    for subj in subjects:
        fpath = _find_file(folder, cond_name, subj)
        sel   = selected_electrodes.get(subj, [])
        if fpath is None or len(sel) == 0: continue
        data = np.load(fpath).mean(0)
        out.append(data[sel, :][:, post_mask].max(1))
    return np.concatenate(out) if out else np.array([])


# Use sliding window EN as sort reference (most words, stable estimate)
ref_mode   = 'sliding_window' if 'sliding_window' in AVAILABLE_MODES else AVAILABLE_MODES[0]
ref_cond   = CONDITIONS[ref_mode]['en']
ref_peaks  = elec_peaks_all(FOLDERS[ref_mode], ref_cond)

if len(ref_peaks) == 0:
    print('No electrode data — skipping Fig 7.')
else:
    sort_idx = np.argsort(ref_peaks)[::-1]
    ranks    = np.arange(1, len(ref_peaks) + 1)

    fig, ax = plt.subplots(figsize=(8, 4.2))

    for mode in AVAILABLE_MODES:
        folder = FOLDERS[mode]
        cond   = CONDITIONS[mode]['en']
        peaks  = elec_peaks_all(folder, cond)
        if len(peaks) == 0: continue
        col = MODE_COL[mode]
        lbl = MODE_LABEL[mode]
        ls  = '--' if mode == 'fasttext' else '-'
        if len(peaks) == len(ref_peaks):
            ax.plot(ranks, peaks[sort_idx], color=col, lw=1.8, ls=ls, label=lbl, zorder=4)
        else:
            s2 = np.argsort(peaks)[::-1]
            ax.plot(np.arange(1, len(peaks)+1), peaks[s2],
                    color=col, lw=1.8, ls=ls, label=f'{lbl} (own rank)', zorder=4)

    ax.axhline(THRESHOLD, color='#E08000', ls=':', lw=1.4,
               label=f'Selection threshold (r={THRESHOLD})', zorder=5)
    ax.axhline(0, color='#CCCCCC', lw=0.6)
    ax.set_xlabel('Electrode rank (sorted by EN sliding-window peak r)', fontsize=10)
    ax.set_ylabel('Peak Pearson r', fontsize=10)
    ax.set_title('Electrode Encoding Profile — All Models, English', fontsize=11)
    ax.legend(fontsize=8, loc='upper right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    save_fig(fig, 'fig5_electrode_profile.png')

## Figure 6 — Pre vs Post Onset

In [ ]:
"""Pre-onset signal speaks to predictive processing.
XGLM (causal) should show less pre-onset correlation than XLM-R (bidirectional)."""

pre_mask2  = (LAGS >= PRE_RANGE[0])  & (LAGS <= PRE_RANGE[1])
post_mask2 = (LAGS >= POST_RANGE[0]) & (LAGS <= POST_RANGE[1])

def window_mean(folder, cond, mask):
    vals = []
    for subj in subjects:
        fpath = _find_file(folder, cond, subj)
        sel   = selected_electrodes.get(subj, [])
        if fpath is None or len(sel) == 0: continue
        data = np.load(fpath).mean(0)
        vals.append(float(data[sel, :][:, mask].mean()))
    return vals


specs_pp = []
for mode in AVAILABLE_MODES:
    folder = FOLDERS[mode]
    cond   = CONDITIONS[mode]['en']
    pre    = window_mean(folder, cond, pre_mask2)
    post   = window_mean(folder, cond, post_mask2)
    if pre and post:
        specs_pp.append((MODE_LABEL[mode], MODE_COL[mode],
                         np.mean(pre), np.mean(post)))

if not specs_pp:
    print('No pre/post data — skipping Fig 8.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    fig.suptitle('Pre- vs Post-Onset Encoding — English Condition',
                 fontsize=12, fontweight='semibold', y=1.02)

    # Scatter
    ax = axes[0]
    all_v = [v for _, _, pv, po in specs_pp for v in [pv, po]]
    lim = (min(all_v) - 0.002, max(all_v) + 0.002)
    ax.plot(lim, lim, color='#CCCCCC', ls='--', lw=0.9, zorder=0)
    for lbl, col, pv, po in specs_pp:
        ax.scatter(pv, po, color=col, s=110, zorder=4,
                   edgecolors='white', lw=0.8)
        ax.annotate(lbl.split('(')[0].strip(), (pv, po),
                    xytext=(6, 4), textcoords='offset points',
                    fontsize=8, color='#444444')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.axvline(0, color='#CCCCCC', lw=0.7); ax.axhline(0, color='#CCCCCC', lw=0.7)
    ax.set_xlabel(f'Mean r: pre-onset ({PRE_RANGE[0]}–{PRE_RANGE[1]} s)', fontsize=10)
    ax.set_ylabel(f'Mean r: post-onset ({POST_RANGE[0]}–{POST_RANGE[1]} s)', fontsize=10)
    ax.set_title('Scatter', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Grouped bar
    ax = axes[1]
    labels_pp = [s[0].split('(')[0].strip() for s in specs_pp]
    pres  = [s[2] for s in specs_pp]
    posts = [s[3] for s in specs_pp]
    x2    = np.arange(len(specs_pp))
    w2    = 0.38
    ax.bar(x2 - w2/2, pres,  w2, label='Pre-onset',
           color='#4477AA', alpha=0.88, edgecolor='white', lw=0.6)
    ax.bar(x2 + w2/2, posts, w2, label='Post-onset',
           color='#EE6677', alpha=0.88, edgecolor='white', lw=0.6)
    ax.set_xticks(x2)
    ax.set_xticklabels(labels_pp, rotation=20, ha='right', fontsize=8.5)
    ax.axhline(0, color='#444444', lw=0.6)
    ax.set_ylabel('Mean Pearson r', fontsize=10)
    ax.set_title('Pre vs Post Comparison', fontsize=10)
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout(w_pad=3)
    save_fig(fig, 'fig6_pre_vs_post_onset.png')

## Figure 7 — Per-Subject Distribution

In [ ]:
"""Shows individual subject variability for key conditions.
One panel per mode"""

# Build peak_data 
peak_data = {}
for mode in AVAILABLE_MODES:
    folder = FOLDERS[mode]
    conds  = CONDITIONS[mode]
    for key in ['en', 'he', 'ar', 'en_he_res', 'en_ar_res', 'noise']:
        tag = f'{mode}_{key}'
        peak_data[tag] = subject_peaks(folder, conds[key])
print(f'peak_data ready: {len(peak_data)} conditions')

fig, axes = plt.subplots(1, len(AVAILABLE_MODES),
                          figsize=(4.5 * len(AVAILABLE_MODES), 4.5),
                          sharey=False)
if len(AVAILABLE_MODES) == 1: axes = [axes]
fig.suptitle('Per-Subject Peak Encoding Accuracy',
             fontsize=12, fontweight='semibold', y=1.02)

rng3 = np.random.default_rng(99)

for ax, mode in zip(axes, AVAILABLE_MODES):
    folder = FOLDERS[mode]
    conds  = CONDITIONS[mode]

    vspecs = [
        ('en',        'EN',          LANG_COL['en']),
        ('he',        'HE',          LANG_COL['he']),
        ('ar',        'AR',          LANG_COL['ar']),
        ('en_he_res', 'EN+HE res',   LANG_COL['he']),
        ('en_ar_res', 'EN+AR res',   LANG_COL['ar']),
        ('noise',     'Noise',       LANG_COL['noise']),
    ]

    vdata  = []
    vlabel = []
    vcolor = []
    for key, lbl, col in vspecs:
        vals = peak_data.get(f'{mode}_{key}', [])
        if len(vals) >= 2:
            vdata.append(vals)
            vlabel.append(lbl)
            vcolor.append(col)

    if not vdata:
        ax.set_visible(False)
        continue

    pos = np.arange(1, len(vdata) + 1)
    vp  = ax.violinplot(vdata, positions=pos, showmedians=True,
                        showextrema=True, widths=0.55)

    for body, col in zip(vp['bodies'], vcolor):
        body.set_facecolor(col)
        body.set_edgecolor('white')
        body.set_alpha(0.50)
        body.set_linewidth(0.8)
    for part in ('cmedians', 'cbars', 'cmins', 'cmaxes'):
        if part in vp:
            vp[part].set_edgecolor('#444444')
            vp[part].set_linewidth(1.2)

    for xi, vals, col in zip(pos, vdata, vcolor):
        jit = rng3.uniform(-0.12, 0.12, len(vals))
        ax.scatter(xi + jit, vals, color=col, s=42, zorder=5,
                   edgecolors='white', lw=0.5)

    ax.axhline(THRESHOLD, color='#E08000', ls=':', lw=1.2,
               label=f'r={THRESHOLD}', zorder=6)
    ax.axhline(0, color='#CCCCCC', lw=0.6)
    ax.set_xticks(pos)
    ax.set_xticklabels(vlabel, fontsize=8.5)
    ax.set_ylabel('Peak Pearson r  (0–2 s)', fontsize=10)
    ax.set_title(MODE_LABEL[mode], fontsize=9.5,
                 color=MODE_COL[mode], fontweight='semibold')
    ax.legend(fontsize=7.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(w_pad=3)
save_fig(fig, 'fig7_per_subject_violin.png')

## Statistical Summary Table

In [ ]:
"""Full table: mean peak r, SEM, p vs noise (FDR), p vs EN same mode (FDR),
Cohen's d vs EN, significance marker."""

all_rows   = []
raw_p_noise, raw_p_en = [], []

for mode in AVAILABLE_MODES:
    noise_vals = peak_data.get(f'{mode}_noise', [])
    en_vals    = peak_data.get(f'{mode}_en',    [])
    for key, display in [
        ('en',       'EN'),
        ('he',       'HE'),
        ('ar',       'AR'),
        ('en_he_res','EN+HE residual'),
        ('en_ar_res','EN+AR residual'),
        ('noise',    'Noise'),
    ]:
        tag  = f'{mode}_{key}'
        vals = peak_data.get(tag, [])
        n    = len(vals)
        if n < 2:
            all_rows.append((mode, display, n, np.nan, np.nan, np.nan, np.nan, np.nan))
            continue
        mean_r = np.mean(vals)
        sem_r  = np.std(vals, ddof=1) / np.sqrt(n)

        # vs noise
        nn = min(n, len(noise_vals))
        if nn >= 3 and key != 'noise':
            _, p_n = stats.ttest_rel(vals[:nn], noise_vals[:nn])
        else:
            p_n = np.nan

        # vs EN same mode
        ne = min(n, len(en_vals))
        if ne >= 3 and key != 'en':
            _, p_e = stats.ttest_rel(vals[:ne], en_vals[:ne])
            d_e    = cohens_d(vals[:ne], en_vals[:ne])
        else:
            p_e, d_e = np.nan, np.nan

        all_rows.append((mode, display, n, mean_r, sem_r, p_n, p_e, d_e))
        if not np.isnan(p_n): raw_p_noise.append(p_n)
        if not np.isnan(p_e): raw_p_en.append(p_e)

corr_noise = iter(fdr(raw_p_noise))
corr_en    = iter(fdr(raw_p_en))

hdr = (f'{"Mode":<16} {"Condition":<18} {"n":>3} {"Mean r":>8} '
       f'{"SEM":>7} {"p_noise":>10} {"p_EN":>10} {"d_EN":>8} {"sig":>5}')
print(hdr)
print('─' * len(hdr))

prev_mode = None
for (mode, disp, n, mr, sem, p_n, p_e, d_e) in all_rows:
    if mode != prev_mode:
        print()
        prev_mode = mode
    pn_str = f'{next(corr_noise):.4f}' if not np.isnan(p_n) else '    —   '
    pe_str = f'{next(corr_en):.4f}'    if not np.isnan(p_e) else '    —   '
    de_str = f'{d_e:.3f}'              if not np.isnan(d_e) else '    —  '
    sg_str = sig_star(float(pe_str))   if pe_str.strip() != '—' else '—'
    mr_str = f'{mr:.4f}'               if not np.isnan(mr)  else '   —   '
    sm_str = f'{sem:.4f}'              if not np.isnan(sem) else '   —   '
    print(f'{mode:<16} {disp:<18} {n:>3} {mr_str:>8} '
          f'{sm_str:>7} {pn_str:>10} {pe_str:>10} {de_str:>8} {sg_str:>5}')

print()
print('p-values FDR-corrected (Benjamini-Hochberg).')
print('sig: ns p≥0.05 | * p<0.05 | ** p<0.01 | *** p<0.001')
print()
print(f'All figures saved to: {os.path.abspath(FIG_DIR)}')